In [1]:
import pandas as pd
from pathlib import Path

In [2]:
indicator_dict = {
    'PIB per capita':'NY.GDP.PCAP.CD',
    'Crecimiento del PIB':'NY.GDP.MKTP.KD.ZG',
    'Rentas del carbon % PIB':'NY.GDP.COAL.RT.ZS',
    'Empleo vulnerable':'SL.EMP.VULN.ZS',
    'Gasto P educacion':'SE.XPD.TOTL.GB.ZS',
    'var IPC':'FP.CPI.TOTL.ZG',
    'TD total':'SL.UEM.TOTL.NE.ZS',
    'TD juvenil':'SL.UEM.1524.NE.ZS', #Estimacion nacional, tambien esta de la OIT
    'TD masculina':'SL.UEM.TOTL.MA.NE.ZS',
    'TD femenina':'SL.UEM.TOTL.FE.NE.ZS',
    'TGP':'SL.TLF.CACT.NE.ZS',
    'PEA':'SL.TLF.TOTL.IN',
    'GINI':'SI.POV.GINI',
    'pobreza multidimensional':'SI.POV.MPUN'
    
}
    
    

# Fetching
Only execute this group of cells once

In [4]:
import wbgapi as wb
wb.series.info(q='poverty')

id,value
SE.LPV.PRIM,Learning poverty: Share of Children at the End-of-Primary age below minimum reading proficiency adjusted by Out-of-School Children (%)
SE.LPV.PRIM.FE,Learning poverty: Share of Female Children at the End-of-Primary age below minimum reading proficiency adjusted by Out-of-School Children (%)
SE.LPV.PRIM.MA,Learning poverty: Share of Male Children at the End-of-Primary age below minimum reading proficiency adjusted by Out-of-School Children (%)
SH_UHC_FH40_FURTHER,"Proportion of population further impoverished due to out-of-pocket health expenditure, based on the societal poverty line (%)"
SH_UHC_FH40_IMPOV,"Proportion of population facing impoverishing out-of-pocket health expenditure, based on the societal poverty line (%)"
SH_UHC_FH40_PUSHED,Proportion of population pushed into poverty (based on the societal poverty line) due to out-of-pocket health expenditure (%)
SI.POV.DDAY,Poverty headcount ratio at $3.00 a day (2021 PPP) (% of population)
SI.POV.GAPS,Poverty gap at $3.00 a day (2021 PPP) (%)
SI.POV.LMIC,Poverty headcount ratio at $4.20 a day (2021 PPP) (% of population)
SI.POV.LMIC.GP,Poverty gap at $4.20 a day (2021 PPP) (%)


In [6]:
dfs = {}
csv_folder = 'csv_files'
for ind in indicator_dict.keys():
    if not Path(f'{csv_folder}/{ind}.parquet').is_file():
        dfs[ind] = wb.data.DataFrame(
            indicator_dict[ind], 
            numericTimeKeys=True, 
            columns='series'
        ).reset_index()
        dfs[ind].to_parquet(f'{csv_folder}/{ind}.parquet')
    else: pass

# preprocessing

In [3]:
dfs = {string.stem: pd.read_parquet(string) for string in Path('csv_files/').glob('*.parquet')}

In [4]:
for df in dfs.keys():
    dfs[df] = dfs[df].rename(columns={k:v for v,k in indicator_dict.items()})

In [5]:
from functools import reduce
merged_df = reduce(lambda x,y: pd.merge(x,y,on=['economy','time']), dfs.values())

In [8]:
merged_df.to_parquet('data.parquet')

In [6]:
merged_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 17490 entries, 0 to 17489
Data columns (total 16 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   economy                   17490 non-null  str    
 1   time                      17490 non-null  int64  
 2   PIB per capita            14745 non-null  float64
 3   var IPC                   11472 non-null  float64
 4   TD total                  6096 non-null   float64
 5   TD juvenil                4298 non-null   float64
 6   TD masculina              5480 non-null   float64
 7   TD femenina               5467 non-null   float64
 8   TGP                       5733 non-null   float64
 9   PEA                       8410 non-null   float64
 10  Crecimiento del PIB       14323 non-null  float64
 11  Rentas del carbon % PIB   10662 non-null  float64
 12  Empleo vulnerable         8176 non-null   float64
 13  Gasto P educacion         5739 non-null   float64
 14  GINI             